# Waymo Open Dataset → NeRF/3DGS 데이터셋 변환 파이프라인 해설

이 노트북은 두 개의 스크립트(`generate_dataset.py`, `data_viewer.py`)를 결합해
**Waymo Open Dataset에서 추출한 카메라 데이터의 원리와 구조를 단계별로 시각화**합니다.

## 다루는 내용
1. **데이터 생성 파이프라인 개요** ─ 어떤 데이터를 왜 추출하는가
2. **Camera Intrinsic 파라미터의 원리** ─ 핀홀 모델, 초점거리, 주점, 왜곡
3. **JSON으로 저장된 Intrinsic 들여다보기**
4. **Intrinsic 시각화** ─ 이미지 평면, 광학 중심, FoV
5. **Camera Extrinsic 파라미터의 원리** ─ 좌표계 사슬 (Camera → Vehicle → World)
6. **JSON으로 저장된 Extrinsic 들여다보기**
7. **Extrinsic 시각화** ─ 3D Frustum, 6-DOF Timeline, Vehicle vs Camera
8. **Depth + Panoptic + Camera 파라미터의 결합 활용**
9. **종합 요약**

## 사전 준비
- `generate_dataset.py`를 한 번 실행하여 `./output/` 디렉토리가 만들어져 있어야 합니다.
- `data_viewer.py`가 이 노트북과 같은 디렉토리에 있어야 합니다 (함수들을 import 해서 사용).


---
## 1. 환경 설정 (Setup)

먼저 필요한 모듈을 import하고, `data_viewer.py`에 정의된 로더/시각화 함수를 그대로 가져옵니다.
이렇게 하면 노트북에서 같은 함수를 다시 작성할 필요 없이 모듈 단일 진실 공급원(single source of truth)으로 활용할 수 있습니다.


In [2]:
# 표준 라이브러리
import os                                           # 경로 처리
import json                                         # JSON 파싱
import numpy as np                                  # 수치 연산

# 시각화 라이브러리
import matplotlib                                   # matplotlib 백엔드 설정
matplotlib.use('module://matplotlib_inline.backend_inline')  # 노트북에서 inline으로 보이게 설정
import matplotlib.pyplot as plt                     # 그래프
from PIL import Image                               # 이미지 로드

# data_viewer.py의 핵심 함수들을 그대로 import
# (코드 중복을 피하면서 '두 스크립트를 합쳐서 보여준다'는 목적을 달성)
from data_viewer import (
    load_extrinsic_json,                  # JSON 형식 Extrinsic 로더
    load_intrinsic_json,                  # JSON 형식 Intrinsic 로더
    load_panoptic_info_json,               # Panoptic 메타정보 로더 (txt)
    rotation_matrix_to_waymo_euler,       # 회전행렬 → 오일러각
    compute_waymo_frustum_corners,        # 카메라 Frustum 모서리 계산
    draw_camera_3d,                       # 3D 카메라/Frustum 그리기
    set_axes_equal_3d,                    # 3D 축 비율 맞추기
    visualize_per_frame_combined,         # 프레임별 통합 시각화
    visualize_intrinsic_on_image,         # Intrinsic 기하학 시각화
    visualize_summary_grid,               # 전체 프레임 요약 격자
    visualize_multiview_3d,               # 3D 멀티뷰
    visualize_topdown_2d,                 # Bird's eye view
    visualize_6dof_timeline,              # 6자유도 타임라인
    visualize_frustums_with_thumbnails,   # Frustum + 썸네일
    visualize_vehicle_vs_camera,          # 차량 vs 카메라 비교
)

# 데이터 루트 경로 설정 (generate_dataset.py 출력 디렉토리)
OUTPUT_ROOT = './output'

# 분석에 사용할 대표 프레임 인덱스 (제일 첫 프레임)
SAMPLE_FRAME = 6
SAMPLE_PREFIX = f'frame_{SAMPLE_FRAME:04d}'

print(f"분석 대상 프레임: {SAMPLE_PREFIX}")
print(f"데이터 경로: {OUTPUT_ROOT}")


분석 대상 프레임: frame_0006
데이터 경로: ./output


---
## 2. 데이터 생성 파이프라인 개요

`generate_dataset.py`는 Waymo TFRecord에서 다음 6가지 산출물을 만듭니다:

| 산출물 | 용도 | 모델/소스 | 형식 |
|---|---|---|---|
| **이미지** | 3D 재구성의 입력 컬러 픽셀 | Waymo 원본 JPEG | `.jpg` |
| **Depth Map** | 재구성 시 깊이 supervision | DepthPro (Apple) | `.npy` (float meter), `.jpg` 시각화 |
| **Panoptic Mask** | 동적 객체 분리, 의미 인식 | Mask2Former (Cityscapes) | `.npy` (segment ID), `.txt` 메타 |
| **Extrinsic** | 카메라가 World 어디서 무엇을 보고 있는가 | Waymo 캘리브레이션 + 차량 포즈 | **`.json`** (의미 키 포함) |
| **Intrinsic** | 픽셀과 광선의 관계 | Waymo 캘리브레이션 | **`.json`** (의미 키 포함) |

### 왜 JSON인가?

이전 버전은 `.txt`로 저장했지만, 다음 이유로 JSON으로 변환했습니다:
- **자기 설명적(self-describing)**: 각 키가 어떤 의미인지 명확히 표시할 수 있다.
- **파싱 안정성**: 정규표현식 없이 표준 라이브러리(`json`)만으로 안정적으로 읽을 수 있다.
- **확장성**: 파생값(FoV, 오프셋 등)을 같이 넣어도 구조가 깨지지 않는다.

### 추출 대상 프레임 정책
시작 프레임 `6`부터 `6`프레임 간격으로 총 `12`프레임 → `[6, 12, 18, ..., 72]`

이 정책의 이유: 인접 프레임은 거의 정지된 듯 비슷하므로, 적당히 띄워야 카메라 베이스라인(baseline)이 충분히 확보되어 3D 재구성이 안정적입니다.


---
## 3. Camera Intrinsic 파라미터의 원리

### 3.1 핀홀 카메라 모델

3D 점 $\mathbf{X}_c = (X, Y, Z)$ (카메라 좌표계 기준)가 이미지 픽셀 $(u, v)$에 어떻게 투영되는지 다음 식으로 모델링합니다:

$$
\begin{bmatrix} u \\ v \\ 1 \end{bmatrix}
\;=\;
\frac{1}{Z}\,
\underbrace{\begin{bmatrix} f_u & 0 & c_u \\ 0 & f_v & c_v \\ 0 & 0 & 1 \end{bmatrix}}_{K\,(\text{intrinsic matrix})}
\begin{bmatrix} X \\ Y \\ Z \end{bmatrix}
$$

각 파라미터의 의미:

| 기호 | 단위 | 의미 |
|---|---|---|
| $f_u, f_v$ | pixel | **초점거리(focal length)**. 카메라의 "줌 정도"를 픽셀 단위로 표현. |
| $c_u, c_v$ | pixel | **주점(principal point)**. 광축이 이미지 평면에 닿는 위치. 이상적으로 이미지 정중앙이지만 실제 렌즈는 미세하게 어긋남. |
| $k_1, k_2, k_3$ | (없음) | **방사 왜곡(radial distortion)** 계수. 렌즈 곡면 때문에 직선이 휘어 보이는 현상. |
| $p_1, p_2$ | (없음) | **접선 왜곡(tangential distortion)** 계수. 렌즈와 센서가 평행하지 않을 때 생김. |

### 3.2 Waymo의 1D 표현 vs 표준 3x3 표현

Waymo는 9원소 1D 배열로 모든 것을 한 번에 저장합니다:
$$\text{intrinsic}_{\text{Waymo}} = [f_u,\; f_v,\; c_u,\; c_v,\; k_1,\; k_2,\; p_1,\; p_2,\; k_3]$$

NeRF/3DGS/OpenCV 같은 표준 라이브러리는 3x3 K 행렬과 5원소 왜곡 벡터를 분리해서 받습니다. `generate_dataset.py`의 `waymo_intrinsic_to_K_matrix()`가 이 변환을 수행하며, 결과를 JSON에 둘 다 담아 저장합니다.

### 3.3 Field of View (FoV)

이미지 폭과 초점거리로부터 수평 FoV를 계산할 수 있습니다:
$$\text{FoV}_\text{horizontal} = 2\,\arctan\!\left(\frac{W}{2 f_u}\right)$$

이 값은 **카메라가 한 프레임에서 얼마나 넓은 각도를 볼 수 있는가**를 의미합니다. JSON의 `derived_quantities`에 미리 계산해 두었습니다.


---
## 4. JSON으로 저장된 Intrinsic 들여다보기

JSON 파일을 raw 텍스트로 한 번 보고, 그 다음 `load_intrinsic_json()`으로 파이썬 dict로 가져와 핵심 값들을 출력합니다.


In [3]:
# 1) Raw JSON 텍스트 출력 (의미 키 구조 확인)
intr_json_path = os.path.join(OUTPUT_ROOT, 'intrinsic', f'{SAMPLE_PREFIX}.json')

with open(intr_json_path, 'r', encoding='utf-8') as f:
    raw_json = f.read()

# 너무 길어지지 않도록 처음 70줄만 출력
print(f"=== {intr_json_path} (첫 70줄) ===\n")
for i, line in enumerate(raw_json.splitlines()[:70], 1):
    print(f"{i:3d}| {line}")
print("...")


=== ./output/intrinsic/frame_0006.json (첫 70줄) ===

  1| {
  2|   "metadata": {
  3|     "data_type": "Camera Intrinsic Parameters",
  4|     "source_dataset": "Waymo Open Dataset",
  5|     "frame_index": 6,
  6|     "camera_name": "FRONT",
  7|     "generated_at": "2026-04-23T03:17:10",
  8|     "description": "Waymo Open Dataset에서 추출한 카메라의 내부 파라미터입니다. Waymo 원본 1D 배열, 표준 3x3 K 행렬, OpenCV 호환 왜곡 계수, 그리고 화각(FoV)을 모두 포함합니다.",
  9|     "distortion_model": "Brown-Conrady (OpenCV 호환)"
 10|   },
 11|   "image_resolution": {
 12|     "width": 1920,
 13|     "height": 1280,
 14|     "unit": "pixels"
 15|   },
 16|   "waymo_raw_intrinsic": {
 17|     "description": "Waymo 원본 형식. 길이 9의 1D 배열로 [f_u, f_v, c_u, c_v, k1, k2, p1, p2, k3] 순서.",
 18|     "format": "1D array of 9 float values",
 19|     "source_field": "frame.context.camera_calibrations[*].intrinsic",
 20|     "as_array": [
 21|       2080.323261452526,
 22|       2080.323261452526,
 23|       971.1680774787271,
 24|       646.494516570

In [5]:
# 2) load_intrinsic_json()으로 파싱하여 핵심 값 표시
intr = load_intrinsic_json(intr_json_path)

# K 행렬과 왜곡 계수, 해상도를 깔끔하게 출력
print(f"이미지 해상도        : {intr['width']} x {intr['height']} px\n")

print("표준 3x3 K 행렬:")
np.set_printoptions(precision=3, suppress=True)
print(intr['K'], "\n")

print(f"왜곡 계수 [k1, k2, p1, p2, k3]:\n  {intr['distortion']}\n")

# JSON에 미리 계산해 둔 파생값 활용
with open(intr_json_path, 'r', encoding='utf-8') as f:
    full = json.load(f)



# derived = full['derived_quantities']
# print("파생 정보 (derived_quantities):")
# for k, v in derived.items():
#     val = v['value']
#     unit = v.get('unit', '')
#     desc = v.get('description', '')
#     print(f"  {k:40s} = {val} [{unit}]")
#     print(f"    └ {desc}")


# 1. 고유 파라미터 상세 정보 출력 (기존 코드의 목적)
print("카메라 파라미터 상세 정보:")
# waymo_raw_intrinsic 안의 as_named_values를 가져옵니다.
named_values = full['waymo_raw_intrinsic']['as_named_values']


for k, v in named_values.items():
    val = v['value']
    unit = v.get('unit', '')
    desc = v.get('description', '')
    print(f"  {k:10s} = {val:<25} [{unit}]")
    print(f"    └ {desc}")

print("\n" + "-"*50 + "\n")

# 2. 진짜 파생 정보인 화각(FoV) 출력
print("파생 정보 (Field of View):")
fov = full['camera_matrix_K']['field_of_view']
print(f"  수평 화각 = {fov['horizontal_deg']:.2f} [{fov['unit']}]")
print(f"  수직 화각 = {fov['vertical_deg']:.2f} [{fov['unit']}]")
print(f"    └ 계산식: {fov['computation']}")


이미지 해상도        : 1920 x 1280 px

표준 3x3 K 행렬:
[[2080.323    0.     971.168]
 [   0.    2080.323  646.495]
 [   0.       0.       1.   ]] 

왜곡 계수 [k1, k2, p1, p2, k3]:
  [ 0.058 -0.394  0.003 -0.     0.   ]



NameError: name 'named_values' is not defined

---
## 5. Intrinsic 시각화

`visualize_intrinsic_on_image()`는 한 프레임의 Intrinsic을 세 패널로 보여줍니다:

1. **이미지 평면 뷰**: 원본 이미지 위에 "이미지 기하 중심"(회색 ✕)과 "주점(광학 중심)"(빨간 +)을 표시. 두 점이 약간 어긋난 것이 보입니다 — 이것이 바로 $c_u \ne W/2$인 이유입니다.
2. **Top-Down 핀홀 도식**: 위에서 본 핀홀 모델. 초점거리, FoV, 센서 평면, 광축이 한눈에 들어옵니다.
3. **수치 정보**: 모든 파라미터의 값과 의미.

함수가 파일로 저장하므로, 저장된 PNG를 노트북에 표시합니다.


In [ ]:
# 시각화를 위한 임시 출력 폴더 만들기
VIS_DIR = './notebook_viz'
os.makedirs(VIS_DIR, exist_ok=True)

# 원본 이미지 로드
img = np.array(Image.open(os.path.join(OUTPUT_ROOT, 'image', f'{SAMPLE_PREFIX}.jpg')))

# Intrinsic 시각화 생성 (PNG로 저장)
intrinsic_png = os.path.join(VIS_DIR, f'{SAMPLE_PREFIX}_intrinsic.png')
visualize_intrinsic_on_image(img, intr, SAMPLE_FRAME, intrinsic_png)

# 노트북에 표시
fig, ax = plt.subplots(figsize=(20, 6))
ax.imshow(Image.open(intrinsic_png))
ax.axis('off')
plt.tight_layout()
plt.show()

print(f"저장 위치: {intrinsic_png}")


---
## 6. Camera Extrinsic 파라미터의 원리

### 6.1 좌표계 사슬

Waymo는 세 개의 좌표계를 사용합니다:

```
[Camera 좌표계]  --(cam2vehicle)-->  [Vehicle 좌표계]  --(vehicle2world)-->  [World 좌표계]
   x: forward                            x: forward                              ENU 기반
   y: left                               y: left                                  글로벌 좌표
   z: up                                 z: up
```

각 좌표계 사이의 변환은 4x4 동차 변환 행렬(homogeneous transform)로 표현됩니다:

$$
T = \begin{bmatrix} R_{3\times 3} & \mathbf{t}_{3\times 1} \\ \mathbf{0}^\top & 1 \end{bmatrix}
$$

- $R$: 3x3 회전 행렬
- $\mathbf{t}$: 3x1 평행이동 벡터

### 6.2 NeRF/3DGS가 원하는 형태: cam2world

대부분의 3D 재구성 라이브러리는 "이 카메라가 World 좌표계에서 어디 있고 어디를 바라보는가"를 표현하는 **cam2world** 행렬을 직접 받습니다. Waymo가 주는 두 행렬을 곱해서 만들 수 있습니다:

$$T_{\text{cam} \to \text{world}} \;=\; T_{\text{vehicle} \to \text{world}} \cdot T_{\text{cam} \to \text{vehicle}}$$

`generate_dataset.py`는 이 합성을 수행하여 세 행렬(cam2vehicle, vehicle2world, cam2world)을 모두 JSON에 저장합니다.

### 6.3 cam2world의 구조 해석

$T_{\text{cam} \to \text{world}}$의 각 부분이 의미하는 것:

| 부분 | 의미 |
|---|---|
| $\mathbf{t} = T[:3, 3]$ | **카메라의 World 좌표계 위치** (3D 점) |
| $R[:, 0]$ (1번째 열) | 카메라 forward 축이 World에서 가리키는 방향 |
| $R[:, 1]$ (2번째 열) | 카메라 left 축이 World에서 가리키는 방향 |
| $R[:, 2]$ (3번째 열) | 카메라 up 축이 World에서 가리키는 방향 |

이 정보로 **3D Frustum** (시야각 피라미드)을 그릴 수 있습니다.


---
## 7. JSON으로 저장된 Extrinsic 들여다보기


In [ ]:
# 1) Raw JSON 텍스트 출력
ext_json_path = os.path.join(OUTPUT_ROOT, 'extrinsic', f'{SAMPLE_PREFIX}.json')

with open(ext_json_path, 'r', encoding='utf-8') as f:
    raw_json = f.read()

print(f"=== {ext_json_path} (첫 60줄) ===\n")
for i, line in enumerate(raw_json.splitlines()[:60], 1):
    print(f"{i:3d}| {line}")
print("...")


In [ ]:
# 2) load_extrinsic_json()으로 파싱하여 세 행렬 모두 출력
ext = load_extrinsic_json(ext_json_path)

print("=== cam2vehicle (Waymo 원본, Camera → Vehicle) ===")
print(ext['cam2vehicle'], "\n")

print("=== vehicle2world (Waymo 원본, Vehicle → World) ===")
print(ext['vehicle2world'], "\n")

print("=== cam2world (NeRF/3DGS 표준, Camera → World) ===")
print(ext['cam2world'], "\n")

# 합성 검증: vehicle2world @ cam2vehicle == cam2world ?
recomputed = ext['vehicle2world'] @ ext['cam2vehicle']
err = np.abs(recomputed - ext['cam2world']).max()
print(f"검증: |vehicle2world @ cam2vehicle - cam2world|_max = {err:.2e}")
print("→ 0에 매우 가까우면 두 행렬의 합성이 정확하게 cam2world를 만들어냈다는 뜻.")


In [ ]:
# 3) cam2world의 카메라 위치/방향 추출
cam2world = ext['cam2world']
camera_pos_world = cam2world[:3, 3]
forward_axis_world = cam2world[:3, 0]   # x 열 = forward
left_axis_world    = cam2world[:3, 1]   # y 열 = left
up_axis_world      = cam2world[:3, 2]   # z 열 = up

print(f"카메라의 World 위치: {camera_pos_world}")
print(f"카메라 Forward 방향: {forward_axis_world}  (||·||={np.linalg.norm(forward_axis_world):.3f})")
print(f"카메라 Left 방향   : {left_axis_world}  (||·||={np.linalg.norm(left_axis_world):.3f})")
print(f"카메라 Up 방향     : {up_axis_world}  (||·||={np.linalg.norm(up_axis_world):.3f})")

# 회전행렬을 오일러 각도(roll, pitch, yaw)로 변환
roll, pitch, yaw = rotation_matrix_to_waymo_euler(cam2world[:3, :3])
print(f"\n오일러 각도 (Waymo 좌표계 기준):")
print(f"  Roll  = {roll:+.2f}°  (전방축 기준 좌우 기울기)")
print(f"  Pitch = {pitch:+.2f}°  (좌우축 기준 위아래 기울기)")
print(f"  Yaw   = {yaw:+.2f}°  (상하축 기준 좌우 회전)")


---
## 8. Extrinsic 시각화 1 — 3D 카메라 Frustum

여러 프레임의 카메라를 모두 World 좌표계에 한꺼번에 띄워보면 **자율주행 차량의 궤적**과 **카메라가 어디를 바라봤는지**를 동시에 볼 수 있습니다.

각 카메라는 작은 피라미드(Frustum)로 표시되며, 색상은 시간 순서를 나타냅니다 (파란색 → 빨간색).


In [ ]:
# 모든 프레임의 extrinsic / intrinsic / 썸네일을 로드
import re

# extrinsic JSON 파일 목록 정렬
extr_files = sorted([f for f in os.listdir(os.path.join(OUTPUT_ROOT, 'extrinsic')) if f.endswith('.json')])
frame_indices = [int(re.search(r'frame_(\d+)', f).group(1)) for f in extr_files]
print(f"발견된 프레임: {frame_indices}")

all_extrinsics, all_intrinsics, all_thumbs = [], [], []
for fidx in frame_indices:
    prefix = f'frame_{fidx:04d}'
    
    # JSON 로드
    e = load_extrinsic_json(os.path.join(OUTPUT_ROOT, 'extrinsic', f'{prefix}.json'))
    e['frame_idx'] = fidx
    all_extrinsics.append(e)
    
    i = load_intrinsic_json(os.path.join(OUTPUT_ROOT, 'intrinsic', f'{prefix}.json'))
    all_intrinsics.append(i)
    
    # 썸네일
    pil = Image.open(os.path.join(OUTPUT_ROOT, 'image', f'{prefix}.jpg'))
    w_new = 400
    thumb = np.array(pil.resize((w_new, int(pil.height * w_new / pil.width)), Image.Resampling.LANCZOS))
    all_thumbs.append(thumb)

print(f"로드 완료: {len(all_extrinsics)}개 프레임의 모든 데이터")


In [ ]:
# 4-way 3D 시각화 생성
multiview_png = os.path.join(VIS_DIR, 'extrinsic_multiview_3d.png')
visualize_multiview_3d(all_extrinsics, all_intrinsics, multiview_png, frustum_depth=4.0)

fig, ax = plt.subplots(figsize=(16, 13))
ax.imshow(Image.open(multiview_png))
ax.axis('off')
plt.tight_layout()
plt.show()


**해석 가이드:**
- 점선은 카메라의 이동 궤적(차량 진행 방향에 가까움).
- 각 피라미드는 해당 프레임에서 카메라가 보고 있던 시야 영역.
- Top View에서는 차량이 어떤 경로로 이동했는지가 가장 명확하게 보입니다.
- Front View에서 모든 카메라가 비슷한 높이에 있다는 것을 확인할 수 있습니다.


---
## 9. Extrinsic 시각화 2 — 6-DOF Timeline

cam2world 행렬에서 **3축 위치(translation)** 와 **3축 회전(roll/pitch/yaw)** 을 시간(프레임)에 따라 그리면, 차량이 어떻게 움직였는지 직관적으로 파악할 수 있습니다.


In [ ]:
timeline_png = os.path.join(VIS_DIR, 'extrinsic_6dof_timeline.png')
visualize_6dof_timeline(all_extrinsics, timeline_png)

fig, ax = plt.subplots(figsize=(16, 8))
ax.imshow(Image.open(timeline_png))
ax.axis('off')
plt.tight_layout()
plt.show()


**해석 가이드:**
- **Translation X (forward)**: 거의 단조 증가 → 차량이 전방으로 일정하게 주행 중.
- **Translation Y (left)**: 변화량이 작으면 직진, 크면 좌/우로 차선 변경 또는 곡선 주행.
- **Translation Z (up)**: 거의 변화 없음 → 평지 주행 (오르막/내리막이라면 변동이 보임).
- **Yaw**: 누적 변화량이 차량의 누적 회전 각도. 직진이면 평탄, 코너링하면 가파르게 변함.


---
## 10. Extrinsic 시각화 3 — Vehicle vs Camera 오프셋

Waymo는 차량 중심(`vehicle2world`)과 카메라 위치(`cam2world`)를 분리해서 제공합니다. 이 둘은 항상 **고정된 장착 오프셋**만큼 떨어져 있어야 합니다 (FRONT 카메라는 차량 앞 윗부분에 장착).

이 시각화로 그 오프셋이 시간에 따라 일정하게 유지되는지 확인할 수 있습니다.


In [ ]:
vc_png = os.path.join(VIS_DIR, 'vehicle_vs_camera.png')
visualize_vehicle_vs_camera(all_extrinsics, vc_png)

fig, ax = plt.subplots(figsize=(18, 8))
ax.imshow(Image.open(vc_png))
ax.axis('off')
plt.tight_layout()
plt.show()

# 오프셋 수치도 출력
offsets = np.array([
    e['cam2world'][:3, 3] - e['vehicle2world'][:3, 3]
    for e in all_extrinsics
])
print("프레임별 (Camera - Vehicle) 오프셋 [m]:")
print(f"  X (forward) 평균: {offsets[:, 0].mean():+.3f},  표준편차: {offsets[:, 0].std():.4f}")
print(f"  Y (left)    평균: {offsets[:, 1].mean():+.3f},  표준편차: {offsets[:, 1].std():.4f}")
print(f"  Z (up)      평균: {offsets[:, 2].mean():+.3f},  표준편차: {offsets[:, 2].std():.4f}")
print("\n→ 표준편차가 매우 작아야 정상 (카메라가 차량에 단단히 고정되어 있다는 증거).")


---
## 11. Image + Depth + Panoptic 통합 시각화

이제 이미지, Depth Map, Panoptic Segmentation을 한 줄에 나란히 보여줍니다. 이 세 가지를 카메라 파라미터(Intrinsic + Extrinsic)와 결합하면 **3D 재구성**과 **동적 객체 마스킹**의 모든 재료가 갖춰집니다.


In [ ]:
# 샘플 프레임의 통합 시각화
sample_depth = np.load(os.path.join(OUTPUT_ROOT, 'depth', f'{SAMPLE_PREFIX}.npy'))
sample_seg = np.load(os.path.join(OUTPUT_ROOT, 'panoptic', f'{SAMPLE_PREFIX}.npy'))
sample_info = load_panoptic_info_json(os.path.join(OUTPUT_ROOT, 'panoptic', f'{SAMPLE_PREFIX}_info.txt'))

combined_png = os.path.join(VIS_DIR, f'{SAMPLE_PREFIX}_combined.png')
visualize_per_frame_combined(img, sample_depth, sample_seg, sample_info, SAMPLE_FRAME, combined_png, max_depth=50.0)

fig, ax = plt.subplots(figsize=(18, 6))
ax.imshow(Image.open(combined_png))
ax.axis('off')
plt.tight_layout()
plt.show()


**세 채널의 역할:**

| 채널 | NeRF/3DGS 학습에서의 역할 |
|---|---|
| **이미지(RGB)** | 픽셀 색상 supervision (가장 기본). |
| **Depth** | 광선(ray)의 도달 깊이를 직접 supervise → 적은 뷰로도 빠르게 수렴. |
| **Panoptic** | 차량/보행자 같은 동적 객체에 마스크를 씌워 **재구성에서 제외** → static scene만 깔끔하게 학습. |

여기에 **Intrinsic**(픽셀 ↔ 광선 매핑)과 **Extrinsic**(광선의 World 출발점/방향)이 결합되어야 비로소 3D 광선 캐스팅이 가능해집니다.


---
## 12. 모든 프레임 요약 그리드

마지막으로 12개 프레임의 이미지/뎁스/판옵틱을 한 장에 모아서 봅니다. 시간의 흐름에 따라 장면이 어떻게 바뀌는지 한눈에 확인할 수 있습니다.


In [ ]:
# 모든 프레임의 image / depth / panoptic 로드
all_images = [np.array(Image.open(os.path.join(OUTPUT_ROOT, 'image', f'frame_{i:04d}.jpg'))) for i in frame_indices]
all_depths = [np.load(os.path.join(OUTPUT_ROOT, 'depth', f'frame_{i:04d}.npy')) for i in frame_indices]
all_segs   = [np.load(os.path.join(OUTPUT_ROOT, 'panoptic', f'frame_{i:04d}.npy')) for i in frame_indices]

summary_png = os.path.join(VIS_DIR, 'summary_grid.png')
visualize_summary_grid(all_images, all_depths, all_segs, frame_indices, summary_png, max_depth=50.0)

fig, ax = plt.subplots(figsize=(20, 6))
ax.imshow(Image.open(summary_png))
ax.axis('off')
plt.tight_layout()
plt.show()


---
## 13. 종합 요약

### 데이터 흐름 한눈에 보기

```
Waymo TFRecord
      │
      ▼
┌─────────────────────────────────────────────┐
│  generate_dataset.py                        │
│   ├─ JPEG 디코딩  → image/*.jpg            │
│   ├─ DepthPro     → depth/*.npy           │
│   ├─ Mask2Former  → panoptic/*.npy + .txt │
│   ├─ extrinsic    → extrinsic/*.json ★    │
│   └─ intrinsic    → intrinsic/*.json ★    │
└─────────────────────────────────────────────┘
      │
      ▼
┌─────────────────────────────────────────────┐
│  data_viewer.py                             │
│   ├─ load_extrinsic_json()  ← JSON 파싱   │
│   ├─ load_intrinsic_json()  ← JSON 파싱   │
│   └─ visualize_*()           ← 시각화     │
└─────────────────────────────────────────────┘
      │
      ▼
   ./visualization/  (PNG 이미지들)
```

### JSON 변환의 이점 (텍스트 → JSON)

| 기준 | 이전 (.txt) | 현재 (.json) |
|---|---|---|
| 파싱 | 정규표현식 + 헤더 매칭 | `json.load()` 한 줄 |
| 의미 표현 | 주석으로 외부 기술 | 키 이름 자체가 의미 |
| 단위/설명 | 파일 상단 주석에만 | 각 값에 `unit`, `description` 부착 |
| 파생값 | 직접 계산 필요 | `derived_quantities`에 미리 저장 |
| 확장 | 기존 줄 깨질 위험 | 새 키 추가만으로 확장 |

### 다음 단계로 활용
이 출력은 다음 파이프라인의 입력이 됩니다:
- **NeRF (instant-ngp, nerfacto)**: cam2world + K + 이미지 → 신경 라디언스 필드 학습
- **3D Gaussian Splatting**: 동일 입력 + 초기 포인트 클라우드 → 가우시안 폭발적 학습
- **Dynamic Scene 분리**: Panoptic 마스크로 차량/보행자 제외 → static 재구성 안정화

---
*이 노트북은 `generate_dataset.py`(JSON 출력)와 `data_viewer.py`(JSON 로드 + 시각화)를 결합하여, Waymo Open Dataset의 카메라 데이터 추출/시각화 파이프라인 전체를 단일 흐름으로 보여줍니다.*
